In [1]:
# ===== CELL 0: ENV FIX (run once) =====
!pip -q install -U --no-cache-dir \
  "numpy==2.2.6" \
  "opencv-python-headless==4.12.0.88" \
  "timm==1.0.22" \
  "kagglehub" \
  "matplotlib"

import numpy as np, timm, torch
print("numpy:", np.__version__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("timm:", timm.__version__)

print("\nIMPORTANT: Restart session/kernel now (Kaggle: Notebook -> Restart session), then run CELL 1.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 233.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 224.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 246.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 339.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 337.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.0 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.8 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requ

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

numpy: 2.0.2
torch: 2.8.0+cu126 | cuda: True
timm: 1.0.22

IMPORTANT: Restart session/kernel now (Kaggle: Notebook -> Restart session), then run CELL 1.


In [2]:
# ===== CELL 1: FULL PIPELINE =====
import os, gc, math, random, copy, json, zipfile
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
import matplotlib.pyplot as plt

# -------------------- Seed --------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
seed_everything(42)

# -------------------- AMP --------------------
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
autocast_ctx = torch.amp.autocast("cuda", dtype=torch.float16) if USE_CUDA else None
scaler = torch.amp.GradScaler("cuda", enabled=USE_CUDA)

def autocast_or_noop():
    if USE_CUDA:
        return torch.amp.autocast("cuda", dtype=torch.float16)
    from contextlib import nullcontext
    return nullcontext()

# -------------------- Config --------------------
class Config:
    # Better/safer default model (less overfitting than base, still strong)
    # Falls back automatically if not found.
    MODEL_CANDIDATES = [
        "swinv2_small_window8_256.ms_in1k",
        "swinv2_tiny_window8_256.ms_in1k",
        "swin_base_patch4_window7_224",
        "swin_small_patch4_window7_224",
    ]

    # Progressive resize
    IMG_SIZE_STAGE12 = 224
    IMG_SIZE_STAGE3  = 256

    NUM_CLASSES = None  # set after scan
    BATCH_SIZE = 8
    NUM_WORKERS = 2
    ACCUM_STEPS = 2

    # Regularization / Aug
    LABEL_SMOOTHING = 0.05
    WEIGHT_DECAY = 0.03
    DROP_PATH_RATE = 0.20
    HEAD_DROPOUT = 0.25

    # Stage lengths
    STAGE1_EPOCHS = 3
    STAGE2_MAX_EPOCHS = 30
    STAGE3_EPOCHS = 10

    # Stage2 plateau sensitivity (bigger = less “tiny improvement”)
    MIN_DELTA_STAGE2 = 1e-3
    PATIENCE_STAGE2 = 4
    MIN_EPOCHS_STAGE2 = 10

    # Stage3 plateau sensitivity
    MIN_DELTA_STAGE3 = 5e-4
    PATIENCE_STAGE3 = 3
    MIN_EPOCHS_STAGE3 = 3

    # LR (discriminative, stage-based)
    STAGE1_BACKBONE_LR = 2e-5
    STAGE1_HEAD_LR     = 8e-4

    STAGE2_LAYER3_LR   = 8e-5
    STAGE2_LAYER2_LR   = 4e-5
    STAGE2_HEAD_LR     = 3e-4

    STAGE3_LR          = 1.2e-5
    STAGE3_HEAD_LR     = 2.5e-5

    CLIP_GRAD_NORM = 1.0

    # Output
    OUT_DIR = Path("/kaggle/working")
    SAVE_PREFIX = "dermnet_swin_progressive"

print("Device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__)

# -------------------- Data (DermNet) --------------------
# Prefer Kaggle attached dataset if exists; else kagglehub download
DATA_ROOT = None
if Path("/kaggle/input/dermnet").exists():
    DATA_ROOT = Path("/kaggle/input/dermnet")
else:
    import kagglehub
    DATA_ROOT = Path(kagglehub.dataset_download("shubhamgoel27/dermnet"))

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"
assert TRAIN_DIR.exists(), f"TRAIN_DIR not found: {TRAIN_DIR}"
assert TEST_DIR.exists(), f"TEST_DIR not found: {TEST_DIR}"

IMG_EXT = {".jpg",".jpeg",".png",".JPG",".JPEG",".PNG"}

def scan_split(root_dir: Path):
    class_names = sorted([d.name for d in root_dir.iterdir() if d.is_dir()])
    label_map = {c:i for i,c in enumerate(class_names)}

    paths = []
    labels = []
    for c in class_names:
        cdir = root_dir / c
        for f in cdir.iterdir():
            if f.is_file() and f.suffix in IMG_EXT:
                paths.append(str(f))
                labels.append(label_map[c])
    return class_names, label_map, np.array(paths), np.array(labels, dtype=np.int64)

class_names, label_map, all_train_paths, all_train_labels = scan_split(TRAIN_DIR)
_, _, test_paths, test_labels = scan_split(TEST_DIR)

Config.NUM_CLASSES = len(class_names)
print("NUM_CLASSES:", Config.NUM_CLASSES, "| train:", len(all_train_paths), "| test:", len(test_paths))

# -------------------- Stratified split (no sklearn) --------------------
def stratified_split(paths, labels, val_ratio=0.10, seed=42):
    rng = np.random.RandomState(seed)
    train_idx = []
    val_idx = []
    num_classes = labels.max() + 1
    for c in range(num_classes):
        idx = np.where(labels == c)[0]
        rng.shuffle(idx)
        n_val = max(1, int(len(idx) * val_ratio))
        val_idx.append(idx[:n_val])
        train_idx.append(idx[n_val:])
    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx

train_idx, val_idx = stratified_split(all_train_paths, all_train_labels, val_ratio=0.10, seed=42)
train_paths, train_labels = all_train_paths[train_idx], all_train_labels[train_idx]
val_paths, val_labels     = all_train_paths[val_idx], all_train_labels[val_idx]
print("split -> train:", len(train_paths), "| val:", len(val_paths))

# -------------------- Enhancements (CLAHE + Color Constancy) --------------------
import cv2

class ShadesOfGrayColorConstancy:
    def __init__(self, p=6, eps=1e-6):
        self.p = p
        self.eps = eps
    def __call__(self, img: Image.Image) -> Image.Image:
        x = np.asarray(img).astype(np.float32) / 255.0
        rgb = np.power(np.mean(np.power(x, self.p), axis=(0,1)), 1.0 / self.p)
        scale = (np.mean(rgb) / (rgb + self.eps))
        x = np.clip(x * scale[None, None, :], 0.0, 1.0)
        return Image.fromarray((x * 255.0).astype(np.uint8))

class CLAHE_LAB:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8,8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    def __call__(self, img: Image.Image) -> Image.Image:
        x = np.asarray(img).astype(np.uint8)  # RGB
        lab = cv2.cvtColor(x, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l2 = self.clahe.apply(l)
        lab2 = cv2.merge([l2, a, b])
        rgb2 = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)
        return Image.fromarray(rgb2)

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def build_transforms(img_size: int, train: bool):
    enh = [
        ShadesOfGrayColorConstancy(p=6),
        CLAHE_LAB(clip_limit=2.0, tile_grid_size=(8,8)),
    ]
    if train:
        return transforms.Compose(enh + [
            transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.15),
            transforms.RandomRotation(degrees=12),
            transforms.RandAugment(num_ops=2, magnitude=9),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
            transforms.RandomErasing(p=0.10, scale=(0.02,0.12), ratio=(0.3,3.3), value="random"),
        ])
    else:
        return transforms.Compose(enh + [
            transforms.Resize(img_size + 32),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])

# -------------------- Dataset --------------------
class SkinDataset(Dataset):
    def __init__(self, paths, labels, transform=None, max_retries=10):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self.max_retries = max_retries

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        for _ in range(self.max_retries):
            p = self.paths[idx]
            y = int(self.labels[idx])
            try:
                img = Image.open(p).convert("RGB")
                if self.transform: img = self.transform(img)
                return img, torch.tensor(y, dtype=torch.long)
            except Exception:
                idx = random.randint(0, len(self.paths)-1)
        raise RuntimeError(f"Too many failed loads. last={p}")

# -------------------- Loaders + Weighted Sampler --------------------
def build_loaders(img_size: int):
    tr_tf = build_transforms(img_size, train=True)
    va_tf = build_transforms(img_size, train=False)

    tr_ds = SkinDataset(train_paths, train_labels, tr_tf)
    va_ds = SkinDataset(val_paths,   val_labels,   va_tf)
    te_ds = SkinDataset(test_paths,  test_labels,  va_tf)

    # class-balanced sampling (helps imbalance)
    counts = np.bincount(train_labels, minlength=Config.NUM_CLASSES)
    class_w = 1.0 / np.sqrt(counts + 1e-6)
    sample_w = class_w[train_labels]
    num_samples = (len(sample_w) // Config.BATCH_SIZE) * Config.BATCH_SIZE
    sampler = WeightedRandomSampler(torch.tensor(sample_w, dtype=torch.double),
                                    num_samples=num_samples, replacement=True)

    tr_loader = DataLoader(tr_ds, batch_size=Config.BATCH_SIZE, sampler=sampler,
                           drop_last=True, num_workers=Config.NUM_WORKERS,
                           pin_memory=USE_CUDA)
    va_loader = DataLoader(va_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
                           num_workers=Config.NUM_WORKERS, pin_memory=USE_CUDA)
    te_loader = DataLoader(te_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
                           num_workers=Config.NUM_WORKERS, pin_memory=USE_CUDA)
    return tr_loader, va_loader, te_loader

train_loader_224, val_loader_224, _ = build_loaders(Config.IMG_SIZE_STAGE12)

# -------------------- MixUp --------------------
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy

def build_mixup(num_classes):
    return Mixup(
        mixup_alpha=0.2,
        cutmix_alpha=1.0,
        prob=0.8,
        switch_prob=0.5,
        mode="batch",
        label_smoothing=Config.LABEL_SMOOTHING,
        num_classes=num_classes,
    )

mixup_fn = build_mixup(Config.NUM_CLASSES)
ce_soft = SoftTargetCrossEntropy()
ce_plain = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)

# -------------------- Model selection + size fix --------------------
def pick_model_name(candidates):
    avail = set(timm.list_models(pretrained=True))
    for name in candidates:
        if name in avail:
            return name
    # if none found, pick first and hope works
    return candidates[0]

MODEL_NAME = pick_model_name(Config.MODEL_CANDIDATES)
print("Chosen model:", MODEL_NAME)

def create_model(num_classes):
    m = timm.create_model(
        MODEL_NAME,
        pretrained=True,
        num_classes=num_classes,
        drop_path_rate=Config.DROP_PATH_RATE,
    )
    # head dropout
    if hasattr(m, "head") and isinstance(m.head, nn.Linear):
        in_f = m.head.in_features
        m.head = nn.Sequential(nn.Dropout(Config.HEAD_DROPOUT), nn.Linear(in_f, num_classes))

    # --- CRITICAL: allow different input sizes for stage3 ---
    # timm has set_input_size + strict_img_size controls in newer versions
    if hasattr(m, "set_input_size"):
        try:
            m.set_input_size(img_size=Config.IMG_SIZE_STAGE12)
        except Exception:
            pass
    if hasattr(m, "patch_embed"):
        if hasattr(m.patch_embed, "strict_img_size"):
            m.patch_embed.strict_img_size = False
        if hasattr(m.patch_embed, "img_size"):
            # disable hard check
            m.patch_embed.img_size = None
    return m

model = create_model(Config.NUM_CLASSES).to(DEVICE)

# -------------------- Trainable stages --------------------
def set_trainable_stage(model, stage: int):
    for p in model.parameters():
        p.requires_grad = False

    # always train head + norms
    for n, p in model.named_parameters():
        if n.startswith("head."):
            p.requires_grad = True
        if "norm" in n:
            p.requires_grad = True

    if stage >= 2:
        # unfreeze last blocks
        for n, p in model.named_parameters():
            if n.startswith("layers.3."):
                p.requires_grad = True
            if n.startswith("layers.2."):
                p.requires_grad = True

    if stage >= 3:
        for p in model.parameters():
            p.requires_grad = True

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Stage {stage}] Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M")

def build_optimizer_stage2(model):
    # discriminative LR: layer3 > layer2 > others ; head separate
    head, l3, l2, other = [], [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("head."):
            head.append(p)
        elif n.startswith("layers.3."):
            l3.append(p)
        elif n.startswith("layers.2."):
            l2.append(p)
        else:
            other.append(p)

    param_groups = []
    if other:
        param_groups.append({"params": other, "lr": Config.STAGE2_LAYER2_LR * 0.25})
    if l2:
        param_groups.append({"params": l2, "lr": Config.STAGE2_LAYER2_LR})
    if l3:
        param_groups.append({"params": l3, "lr": Config.STAGE2_LAYER3_LR})
    if head:
        param_groups.append({"params": head, "lr": Config.STAGE2_HEAD_LR})

    return optim.AdamW(param_groups, weight_decay=Config.WEIGHT_DECAY)

def build_optimizer_stage1(model):
    head_params, backbone_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if n.startswith("head.") else backbone_params).append(p)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": Config.STAGE1_BACKBONE_LR})
    if head_params:
        groups.append({"params": head_params, "lr": Config.STAGE1_HEAD_LR})
    return optim.AdamW(groups, weight_decay=Config.WEIGHT_DECAY)

def build_optimizer_stage3(model):
    head_params, backbone_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if n.startswith("head.") else backbone_params).append(p)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": Config.STAGE3_LR})
    if head_params:
        groups.append({"params": head_params, "lr": Config.STAGE3_HEAD_LR})
    return optim.AdamW(groups, weight_decay=Config.WEIGHT_DECAY)

# -------------------- Metrics (no sklearn) --------------------
def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def f1_from_cm(cm):
    eps = 1e-12
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    support = cm.sum(axis=1).astype(np.float64)
    f1_macro = np.mean(f1)
    f1_weighted = (f1 * support).sum() / (support.sum() + eps)
    return f1_macro, f1_weighted

# -------------------- Train / Eval loops --------------------
def train_one_epoch(model, loader, optimizer, criterion, mixup=None):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total, correct = 0, 0
    running_loss = 0.0

    for step, (x, y) in enumerate(tqdm(loader, desc="TRAIN", leave=False)):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        use_mix = (mixup is not None) and (x.size(0) % 2 == 0)
        if use_mix:
            x, y_soft = mixup(x, y)
            hard_y = y_soft.argmax(dim=1)
        else:
            y_soft = y
            hard_y = y

        with autocast_or_noop():
            logits = model(x)
            loss = criterion(logits, y_soft) / Config.ACCUM_STEPS

        if USE_CUDA:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % Config.ACCUM_STEPS == 0:
            if USE_CUDA:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.CLIP_GRAD_NORM)

            if USE_CUDA:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        bs = hard_y.size(0)
        running_loss += loss.item() * bs * Config.ACCUM_STEPS
        pred = logits.argmax(dim=1)
        correct += (pred == hard_y).sum().item()
        total += bs

    return running_loss / max(1,total), 100.0 * correct / max(1,total)

@torch.inference_mode()
def evaluate(model, loader, criterion):
    model.eval()
    total, correct = 0, 0
    running_loss = 0.0
    all_pred = []
    all_true = []

    for x, y in tqdm(loader, desc="EVAL", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with autocast_or_noop():
            logits = model(x)
            loss = criterion(logits, y)
        bs = y.size(0)
        running_loss += loss.item() * bs
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += bs

        all_pred.append(pred.detach().cpu().numpy())
        all_true.append(y.detach().cpu().numpy())

    all_pred = np.concatenate(all_pred)
    all_true = np.concatenate(all_true)
    cm = confusion_matrix_np(all_true, all_pred, Config.NUM_CLASSES)
    f1_macro, f1_weighted = f1_from_cm(cm)

    return (running_loss / max(1,total),
            100.0 * correct / max(1,total),
            f1_macro, f1_weighted, cm)

# -------------------- Plot helpers --------------------
def plot_curves(history, out_path):
    epochs = [h["epoch"] for h in history]
    tr_loss = [h["train_loss"] for h in history]
    va_loss = [h["val_loss"] for h in history]
    tr_acc = [h["train_acc"] for h in history]
    va_acc = [h["val_acc"] for h in history]

    plt.figure()
    plt.plot(epochs, tr_loss, label="train_loss")
    plt.plot(epochs, va_loss, label="val_loss")
    plt.legend(); plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Loss Curves")
    plt.tight_layout()
    plt.savefig(out_path / "loss_curves.png", dpi=160)
    plt.close()

    plt.figure()
    plt.plot(epochs, tr_acc, label="train_acc")
    plt.plot(epochs, va_acc, label="val_acc")
    plt.legend(); plt.xlabel("epoch"); plt.ylabel("acc (%)"); plt.title("Accuracy Curves")
    plt.tight_layout()
    plt.savefig(out_path / "acc_curves.png", dpi=160)
    plt.close()

def plot_confusion_matrix(cm, class_names, out_file, normalize=True, topk_labels=23):
    cm_show = cm.astype(np.float64)
    if normalize:
        row_sum = cm_show.sum(axis=1, keepdims=True) + 1e-12
        cm_show = cm_show / row_sum

    # show all classes by default; if too crowded, you can set topk_labels
    n = min(topk_labels, cm.shape[0])
    cm_show = cm_show[:n, :n]
    labels = class_names[:n]

    plt.figure(figsize=(10, 8))
    plt.imshow(cm_show, interpolation="nearest")
    plt.title("Confusion Matrix" + (" (normalized)" if normalize else ""))
    plt.colorbar()
    ticks = np.arange(n)
    plt.xticks(ticks, labels, rotation=90, fontsize=7)
    plt.yticks(ticks, labels, fontsize=7)
    plt.tight_layout()
    plt.savefig(out_file, dpi=180)
    plt.close()

def denorm(x):
    x = x.copy()
    for c in range(3):
        x[c] = x[c] * STD[c] + MEAN[c]
    return np.clip(x, 0, 1)

@torch.inference_mode()
def visualize_batch(loader, out_file, max_images=12):
    x, y = next(iter(loader))
    x = x[:max_images].cpu().numpy()
    y = y[:max_images].cpu().numpy()
    n = len(x)
    cols = 4
    rows = int(math.ceil(n / cols))
    plt.figure(figsize=(12, 3*rows))
    for i in range(n):
        plt.subplot(rows, cols, i+1)
        img = denorm(x[i])
        plt.imshow(np.transpose(img, (1,2,0)))
        plt.title(class_names[int(y[i])], fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_file, dpi=160)
    plt.close()

# -------------------- Stage runner (plateau handling) --------------------
history = []
best_val_loss = float("inf")
best_state = None
best_epoch = -1
global_epoch = 0

def run_stage(stage_id, train_loader, val_loader, optimizer_builder, epochs,
              use_mixup, min_delta, patience, min_epochs, set_input_size_to=None):
    global model, best_val_loss, best_state, best_epoch, global_epoch, history

    # stage-specific input size update
    if set_input_size_to is not None:
        if hasattr(model, "set_input_size"):
            try:
                model.set_input_size(img_size=set_input_size_to)
            except Exception:
                pass
        if hasattr(model, "patch_embed"):
            if hasattr(model.patch_embed, "strict_img_size"):
                model.patch_embed.strict_img_size = False
            if hasattr(model.patch_embed, "img_size"):
                model.patch_embed.img_size = None

    set_trainable_stage(model, stage_id)
    optimizer = optimizer_builder(model)

    # cosine schedule per stage
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0.0)

    mix = mixup_fn if use_mixup else None
    crit = ce_soft if use_mixup else ce_plain

    no_improve = 0
    for e in range(epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, crit, mixup=mix)
        va_loss, va_acc, va_f1m, va_f1w, va_cm = evaluate(model, val_loader, ce_plain)

        # log
        global_epoch += 1
        lrs = [pg["lr"] for pg in optimizer.param_groups]
        history.append({
            "epoch": global_epoch,
            "stage": stage_id,
            "train_loss": tr_loss,
            "train_acc": tr_acc,
            "val_loss": va_loss,
            "val_acc": va_acc,
            "val_f1_macro": float(va_f1m),
            "val_f1_weighted": float(va_f1w),
            "lrs": lrs
        })

        print(f"Epoch {global_epoch:03d} | Stage {stage_id} | "
              f"train {tr_loss:.4f}/{tr_acc:.2f}% | "
              f"val {va_loss:.4f}/{va_acc:.2f}% | "
              f"F1w {va_f1w:.4f} | lrs {[f'{x:.2e}' for x in lrs]}")

        # best tracking
        if va_loss < best_val_loss - min_delta:
            best_val_loss = va_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = global_epoch
            no_improve = 0
            print(f"  -> BEST updated: val_loss={best_val_loss:.4f} @ epoch {best_epoch}")
        else:
            no_improve += 1
            print(f"  -> no improve ({no_improve}/{patience})")

        scheduler.step()

        if global_epoch >= (Config.STAGE1_EPOCHS + Config.STAGE2_MAX_EPOCHS + Config.STAGE3_EPOCHS):
            break

        if e+1 >= min_epochs and no_improve >= patience:
            print(f"[Stage {stage_id}] Early stop at local epoch {e+1}")
            break

# -------------------- Run stages --------------------
# Stage 1 (224, mixup on): head+norm only
run_stage(
    stage_id=1,
    train_loader=train_loader_224,
    val_loader=val_loader_224,
    optimizer_builder=lambda m: build_optimizer_stage1(m),
    epochs=Config.STAGE1_EPOCHS,
    use_mixup=True,
    min_delta=1e-4,
    patience=99,
    min_epochs=99,
    set_input_size_to=Config.IMG_SIZE_STAGE12
)

# Stage 2 (224, mixup on): unfreeze layers.2-3 with discriminative LR, stronger plateau detection
run_stage(
    stage_id=2,
    train_loader=train_loader_224,
    val_loader=val_loader_224,
    optimizer_builder=lambda m: build_optimizer_stage2(m),
    epochs=Config.STAGE2_MAX_EPOCHS,
    use_mixup=True,
    min_delta=Config.MIN_DELTA_STAGE2,
    patience=Config.PATIENCE_STAGE2,
    min_epochs=Config.MIN_EPOCHS_STAGE2,
    set_input_size_to=Config.IMG_SIZE_STAGE12
)

# Stage 3 (256, mixup off): full fine-tune at higher res (fixes your assert)
train_loader_256, val_loader_256, test_loader_256 = build_loaders(Config.IMG_SIZE_STAGE3)

# start from best
if best_state is not None:
    model.load_state_dict(best_state)

run_stage(
    stage_id=3,
    train_loader=train_loader_256,
    val_loader=val_loader_256,
    optimizer_builder=lambda m: build_optimizer_stage3(m),
    epochs=Config.STAGE3_EPOCHS,
    use_mixup=False,
    min_delta=Config.MIN_DELTA_STAGE3,
    patience=Config.PATIENCE_STAGE3,
    min_epochs=Config.MIN_EPOCHS_STAGE3,
    set_input_size_to=Config.IMG_SIZE_STAGE3
)

# load best at end
if best_state is not None:
    model.load_state_dict(best_state)

# -------------------- Final eval on TEST (with simple TTA flips) --------------------
@torch.inference_mode()
def predict_tta(model, loader):
    model.eval()
    all_pred, all_true = [], []
    for x, y in tqdm(loader, desc="TEST(TTA)", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        with autocast_or_noop():
            l0 = model(x)
            l1 = model(torch.flip(x, dims=[3]))  # hflip
            l2 = model(torch.flip(x, dims=[2]))  # vflip
            logits = (l0 + l1 + l2) / 3.0
        pred = logits.argmax(dim=1).cpu().numpy()
        all_pred.append(pred)
        all_true.append(y.numpy())
    all_pred = np.concatenate(all_pred)
    all_true = np.concatenate(all_true)
    cm = confusion_matrix_np(all_true, all_pred, Config.NUM_CLASSES)
    acc = (all_true == all_pred).mean() * 100.0
    f1m, f1w = f1_from_cm(cm)
    return acc, f1m, f1w, cm

test_acc, test_f1m, test_f1w, test_cm = predict_tta(model, test_loader_256)

print("\n" + "="*70)
print("BEST EPOCH:", best_epoch)
print(f"BEST VAL LOSS: {best_val_loss:.4f}")
print(f"TEST ACC (TTA): {test_acc:.2f}% | F1w: {test_f1w:.4f} | F1m: {test_f1m:.4f}")
print("="*70)

# -------------------- Visualizations --------------------
Config.OUT_DIR.mkdir(parents=True, exist_ok=True)

# sample augmented batch
visualize_batch(train_loader_224, Config.OUT_DIR / "augmented_batch_stage12.png", max_images=12)

# curves
plot_curves(history, Config.OUT_DIR)

# confusion matrices
# compute val cm for best model quickly
_, _, _, _, val_cm = evaluate(model, val_loader_256, ce_plain)
plot_confusion_matrix(val_cm, class_names, Config.OUT_DIR / "confusion_val.png", normalize=True)
plot_confusion_matrix(test_cm, class_names, Config.OUT_DIR / "confusion_test.png", normalize=True)

print("Saved plots to:", str(Config.OUT_DIR))

# -------------------- Save model + metadata --------------------
ckpt_path = Config.OUT_DIR / f"{Config.SAVE_PREFIX}_best.pth"
payload = {
    "model_name": MODEL_NAME,
    "num_classes": Config.NUM_CLASSES,
    "class_names": class_names,
    "label_map": label_map,
    "best_epoch": best_epoch,
    "best_val_loss": float(best_val_loss),
    "history": history,
    "state_dict": model.state_dict(),
    "img_size_stage12": Config.IMG_SIZE_STAGE12,
    "img_size_stage3": Config.IMG_SIZE_STAGE3,
}
torch.save(payload, ckpt_path)
print("Checkpoint saved:", ckpt_path)

# -------------------- Zip everything for Kaggle download --------------------
zip_path = Config.OUT_DIR / f"{Config.SAVE_PREFIX}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ckpt_path, arcname=ckpt_path.name)
    for fname in ["loss_curves.png", "acc_curves.png", "confusion_val.png", "confusion_test.png", "augmented_batch_stage12.png"]:
        f = Config.OUT_DIR / fname
        if f.exists():
            z.write(f, arcname=f.name)
    # also store a small json summary
    summary = {
        "model_name": MODEL_NAME,
        "best_epoch": best_epoch,
        "best_val_loss": float(best_val_loss),
        "test_acc_tta": float(test_acc),
        "test_f1_weighted": float(test_f1w),
        "test_f1_macro": float(test_f1m),
    }
    summary_path = Config.OUT_DIR / "summary.json"
    with open(summary_path, "w") as fp:
        json.dump(summary, fp, indent=2)
    z.write(summary_path, arcname=summary_path.name)

print("ZIP saved:", zip_path)
print("\nKaggle'da indirmek için: sağdaki 'Output' panelinden bu dosyayı indir ->", zip_path.name)

# cleanup
gc.collect()
if USE_CUDA:
    torch.cuda.empty_cache()


Device: cuda | torch: 2.8.0+cu126 | timm: 1.0.22
NUM_CLASSES: 23 | train: 15557 | test: 4002
split -> train: 14011 | val: 1546
Chosen model: swinv2_small_window8_256.ms_in1k


model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

[Stage 1] Trainable: 0.06M / 48.98M


Epoch 001 | Stage 1 | train 2.7380/22.93% | val 2.4892/29.17% | F1w 0.2837 | lrs ['2.00e-05', '8.00e-04']
  -> BEST updated: val_loss=2.4892 @ epoch 1


Epoch 002 | Stage 1 | train 2.5368/29.73% | val 2.3328/34.80% | F1w 0.3357 | lrs ['1.50e-05', '6.00e-04']
  -> BEST updated: val_loss=2.3328 @ epoch 2


...

Epoch 036 | Stage 3 | train 0.4458/97.08% | val 1.2836/71.60% | F1w 0.7151 | lrs ['1.09e-05', '2.26e-05']
  -> no improve (1/3)


Epoch 037 | Stage 3 | train 0.4436/97.06% | val 1.2803/71.47% | F1w 0.7127 | lrs ['9.53e-06', '1.98e-05']
  -> no improve (2/3)


Epoch 038 | Stage 3 | train 0.4393/96.96% | val 1.2920/71.73% | F1w 0.7163 | lrs ['7.85e-06', '1.64e-05']
  -> no improve (3/3)
[Stage 3] Early stop at local epoch 5



BEST EPOCH: 35
BEST VAL LOSS: 1.2597
TEST ACC (TTA): 74.29% | F1w: 0.7423 | F1m: 0.7169


Saved plots to: /kaggle/working
Checkpoint saved: /kaggle/working/dermnet_swin_progressive_best.pth
ZIP saved: /kaggle/working/dermnet_swin_progressive_outputs.zip

Kaggle'da indirmek için: sağdaki 'Output' panelinden bu dosyayı indir -> dermnet_swin_progressive_outputs.zip
